In [2]:
# ✅ 1. モジュールと関数の準備
import os
import json
from train import AgentFactory, Env_Geister
from geister_game import GeisterGame
import pandas as pd
from collections import defaultdict

# Elo関連
INITIAL_ELO = 1500
K = 32

def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def update_elo(rating_a, rating_b, score_a, k=K):
    expected_a = expected_score(rating_a, rating_b)
    expected_b = expected_score(rating_b, rating_a)
    new_rating_a = rating_a + k * (score_a - expected_a)
    new_rating_b = rating_b + k * ((1 - score_a) - expected_b)
    return new_rating_a, new_rating_b

In [11]:
# ✅ 2. エージェントの読み込み
agent_info_list = [
    {"id": "Agent1", "config": "models_geister_agentA/eps3000/config.json", "weights": "models_geister_agentA/eps3000/weights.pth"},
    {"id": "Agent2", "config": "trained_models/threads/CQCNN_AngleEmbedding_EfficientSU2_4qbits_4feat_minigeister/CQCNN_AngleEmbedding_EfficientSU2_4qbits_4feat_minigeister_B_ep3000_config.json", "weights": "trained_models/threads/CQCNN_AngleEmbedding_EfficientSU2_4qbits_4feat_minigeister/CQCNN_AngleEmbedding_EfficientSU2_4qbits_4feat_minigeister_B_ep3000_weights.pth"},
    {"id": "Agent3", "config": "trained_models/threads/CQCNN_ZFeatureMap_RealAmplitudes_4qbits_4feat_minigeister/CQCNN_ZFeatureMap_RealAmplitudes_4qbits_4feat_minigeister_A_ep3000_config.json", "weights": "trained_models/threads/CQCNN_ZFeatureMap_RealAmplitudes_4qbits_4feat_minigeister/CQCNN_ZFeatureMap_RealAmplitudes_4qbits_4feat_minigeister_A_ep3000_weights.pth"},
    {"id": "Agent4", "config": "trained_models/threads/CQCNN_ZZFeatureMap_EfficientSU2_4qbits_4feat_minigeister/CQCNN_ZZFeatureMap_EfficientSU2_4qbits_4feat_minigeister_B_ep3000_config.json", "weights": "trained_models/threads/CQCNN_ZZFeatureMap_EfficientSU2_4qbits_4feat_minigeister/CQCNN_ZZFeatureMap_EfficientSU2_4qbits_4feat_minigeister_B_ep3000_weights.pth"},
]


agents = {}
for info in agent_info_list:
    with open(info["config"], "r") as f:
        cfg = json.load(f)
    agent = AgentFactory.create_cqc_agent("A", GeisterGame(board_size=4, num_ghosts_per_player=2), cfg, info["weights"])
    agents[info["id"]] = agent

In [ ]:
NUM_MATCHES_PER_PAIR = 5
elo_ratings = defaultdict(lambda: INITIAL_ELO)

for id_a, agent_a in agents.items():
    for id_b, agent_b in agents.items():
        if id_a == id_b:
            continue

        for _ in range(NUM_MATCHES_PER_PAIR):  # 各ペアで5回対戦
            game = GeisterGame(board_size=4, num_ghosts_per_player=2)
            agent_a.player_id, agent_b.player_id = "A", "B"
            agent_a.game, agent_b.game = game, game

            env = Env_Geister(agent_a, agent_b, game)
            winner, _, _ = env.play_one_game_with_log()

            if winner == "A":
                elo_ratings[id_a], elo_ratings[id_b] = update_elo(elo_ratings[id_a], elo_ratings[id_b], 1)
            elif winner == "B":
                elo_ratings[id_a], elo_ratings[id_b] = update_elo(elo_ratings[id_a], elo_ratings[id_b], 0)
            else:
                elo_ratings[id_a], elo_ratings[id_b] = update_elo(elo_ratings[id_a], elo_ratings[id_b], 0.5)


In [22]:
# ✅ 4. 結果の表示
df = pd.DataFrame([
    {"Model": model, "EloRating": round(score, 2)}
    for model, score in elo_ratings.items()
]).sort_values(by="EloRating", ascending=False)
df.reset_index(drop=True, inplace=True)
df

,Model,EloRating
0,Agent2,1583.37
1,Agent1,1501.82
2,Agent3,1465.37
3,Agent4,1449.44


In [1]:
from train import CNNAgent_Geister, run_geister_cqcnn_training_until_epsilon
from geister_game import GeisterGame
from model_cqcnn import dev_qnn_global


run_geister_cqcnn_training_until_epsilon(
    epsilon_threshold=0.05,
    n_qbits=4,
    cnn_out_feat=4,
    game_instance=GeisterGame(board_size=4, num_ghosts_per_player=2),
    model_save_dir="./trained_models/cqcnn_angle_realamp",
    name_agent1="CQCNN_AngleEmbed_RealAmp_A",
    name_agent2="CQCNN_AngleEmbed_RealAmp_B"
)



--- Training CQCNN Agent until epsilon ≤ 0.05 ---
Initialized QNN device: lightning.qubit with 4 qubits.
[Episode 100] epsilon A: 0.4903, epsilon B: 0.4903
💾 Saved weights: CQCNN_AngleEmbed_RealAmp_A_ep100_weights.pth
📝 Saved config: CQCNN_AngleEmbed_RealAmp_A_ep100_config.json
💾 Saved weights: CQCNN_AngleEmbed_RealAmp_B_ep100_weights.pth
📝 Saved config: CQCNN_AngleEmbed_RealAmp_B_ep100_config.json
[Episode 200] epsilon A: 0.4808, epsilon B: 0.4808
💾 Saved weights: CQCNN_AngleEmbed_RealAmp_A_ep200_weights.pth
📝 Saved config: CQCNN_AngleEmbed_RealAmp_A_ep200_config.json
💾 Saved weights: CQCNN_AngleEmbed_RealAmp_B_ep200_weights.pth
📝 Saved config: CQCNN_AngleEmbed_RealAmp_B_ep200_config.json
[Episode 300] epsilon A: 0.4715, epsilon B: 0.4715
💾 Saved weights: CQCNN_AngleEmbed_RealAmp_A_ep300_weights.pth
📝 Saved config: CQCNN_AngleEmbed_RealAmp_A_ep300_config.json
💾 Saved weights: CQCNN_AngleEmbed_RealAmp_B_ep300_weights.pth
📝 Saved config: CQCNN_AngleEmbed_RealAmp_B_ep300_config.json
[Epi

KeyboardInterrupt: 